# 🦀 Day 7: Mini-Project — Async Chat Server (TCP)

---

## Project Overview

Build a **multi-client TCP chat server**: clients connect, send messages, and the server broadcasts each message to all other connected clients.

- **tokio::net::TcpListener** — accept incoming connections
- **tokio::net::TcpStream** — per-client connection
- **broadcast channel** — fan-out messages to all clients
- **Newline-delimited protocol** — simple framing

In [ ]:
:dep tokio = { version = "1", features = ["full"] }

use tokio::io::{AsyncBufReadExt, AsyncWriteExt, BufReader};
use tokio::net::{TcpListener, TcpStream};
use tokio::sync::broadcast;

## Server Architecture

```
                    ┌─────────────┐
    Client A ──────►│             │
    Client B ──────►│   Server    │──────► broadcast channel
    Client C ──────►│             │           │
                    └─────────────┘           ▼
                                    Each client gets a copy
```

## Client Handler Function

Each client runs in its own task: read lines, broadcast to others.

In [ ]:
async fn handle_client(
    mut socket: TcpStream,
    tx: broadcast::Sender<String>,
    mut rx: broadcast::Receiver<String>,
) {
    let (reader, mut writer) = socket.split();
    let mut reader = BufReader::new(reader);
    let mut line = String::new();

    loop {
        tokio::select! {
            result = reader.read_line(&mut line) => {
                match result {
                    Ok(0) => break,
                    Ok(_) => {
                        let msg = line.trim().to_string();
                        if !msg.is_empty() {
                            let _ = tx.send(msg.clone());
                        }
                        line.clear();
                    }
                    Err(_) => break,
                }
            }
            result = rx.recv() => {
                match result {
                    Ok(msg) => {
                        let _ = writer.write_all(format!("{}\n", msg).as_bytes()).await;
                        let _ = writer.flush().await;
                    }
                    Err(_) => break,
                }
            }
        }
    }
}

## Full Working Server

Accept connections, spawn a task per client, share broadcast channel.

In [ ]:
async fn run_server() {
    let listener = TcpListener::bind("127.0.0.1:8080").await.unwrap();
    println!("Chat server listening on 127.0.0.1:8080");

    let (tx, _) = broadcast::channel::<String>(32);

    loop {
        let (socket, addr) = listener.accept().await.unwrap();
        println!("Client connected: {:?}", addr);

        let tx = tx.clone();
        let rx = tx.subscribe();

        tokio::spawn(async move {
            handle_client(socket, tx, rx).await;
        });
    }
}

In [ ]:
// Run the server (in background for demo — in practice you'd run separately)
// Uncomment to test; use multiple terminals with: echo "hello" | nc 127.0.0.1 8080

// let rt = tokio::runtime::Runtime::new().unwrap();
// rt.spawn(run_server());
// rt.block_on(tokio::time::sleep(tokio::time::Duration::from_secs(5)));

println!("Server code ready. Run in separate process to test.");

## Message Framing: Newline-Delimited

Each message ends with `\n`. `read_line()` reads until newline. Simple and works with `nc` (netcat) for testing.

## Graceful Shutdown (Concept)

For production: use `tokio::select!` with a shutdown signal (e.g. `tokio::signal::ctrl_c()`). When received, stop accepting new connections and drain existing clients.

## 🏋️ Exercise: Add Features

Extend the chat server with one or more:

- **Nicknames**: First message from client = nickname; prefix broadcasts with `[nick]: `
- **Private messages**: Parse `/msg user message` and send only to that user
- **Join/leave announcements**: Broadcast when a client connects or disconnects

In [ ]:
// YOUR CODE HERE
// Implement one of: nicknames, private messages, or join/leave announcements

## 🎯 Key Takeaways

1. **TcpListener::bind** + **accept()** — accept incoming connections
2. **TcpStream** — split into reader/writer for concurrent read/write
3. **broadcast channel** — fan-out messages to all subscribers
4. **BufReader + read_line** — newline-delimited framing
5. **tokio::spawn** — one task per client
6. **tokio::select!** — wait on multiple async operations

---

**Week 14 Complete!** Next week: HTTP & Web 🌐